# OpenEvolve en Kaggle — SHELL: descubrir una heurística de bin-packing

Caso concreto ya cargado: **bin-packing** — empaquetar una lista de ítems
en la menor cantidad posible de contenedores de capacidad fija.

Baseline: First-Fit ingenuo, sin ordenar (a propósito débil, para dejar
margen real de mejora). El evaluador mide SOLO el resultado (contenedores
usados vs. la cota inferior teórica) — nunca instrumenta operaciones
internas, así que no hay atajo de "hacer trampa" con el código: cualquier
forma de mejorar el score es, por construcción, una mejora real del
empaquetamiento.

Este cuaderno sigue siendo un shell reutilizable: los puntos donde
cambiarías algo para adaptarlo a otro problema están marcados con:

```
# ============ CAMBIAR AQUI (N) ============
```

In [14]:
# 0. Verificar GPUs
!nvidia-smi --query-gpu=index,name,memory.total --format=csv

index, name, memory.total [MiB]
0, Tesla T4, 15360 MiB
1, Tesla T4, 15360 MiB


In [15]:
# 1. Dependencia que a veces falta en el entorno de Kaggle
!apt-get update -qq && apt-get install -y -qq zstd

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


In [16]:
# 2. Instalar Ollama
!curl -fsSL https://ollama.com/install.sh | sh

>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [17]:
# 3. Levantar Ollama (una sola instancia, Ollama reparte entre las GPUs solo)
import subprocess, os, time, requests

env = os.environ.copy()
env["OLLAMA_KEEP_ALIVE"] = "-1"
ollama_proc = subprocess.Popen(["ollama", "serve"], env=env)

for _ in range(30):
    try:
        requests.get("http://127.0.0.1:11434")
        print("Ollama arriba.")
        break
    except requests.exceptions.ConnectionError:
        time.sleep(1)

time=2026-08-13T02:53:30.476Z level=INFO source=routes.go:1933 msg="server config" env="map[CUDA_VISIBLE_DEVICES: GGML_VK_VISIBLE_DEVICES: GPU_DEVICE_ORDINAL: HIP_VISIBLE_DEVICES: HSA_OVERRIDE_GFX_VERSION: HTTPS_PROXY: HTTP_PROXY: LLAMA_ARG_FIT: LLAMA_ARG_FIT_TARGET: NO_PROXY: OLLAMA_CONTEXT_LENGTH:0 OLLAMA_DEBUG:INFO OLLAMA_DEBUG_LOG_REQUESTS:false OLLAMA_EDITOR: OLLAMA_FLASH_ATTENTION:false OLLAMA_GO_TEMPLATE:true OLLAMA_GPU_OVERHEAD:0 OLLAMA_HOST:http://127.0.0.1:11434 OLLAMA_IGPU_ENABLE: OLLAMA_KEEP_ALIVE:2562047h47m16.854775807s OLLAMA_KV_CACHE_TYPE: OLLAMA_LLM_LIBRARY: OLLAMA_LOAD_TIMEOUT:5m0s OLLAMA_MAX_LOADED_MODELS:0 OLLAMA_MAX_QUEUE:512 OLLAMA_MAX_TRANSFER_STREAMS:4 OLLAMA_MODELS:/root/.ollama/models OLLAMA_NOHISTORY:false OLLAMA_NOPRUNE:false OLLAMA_NO_CLOUD:false OLLAMA_NUM_PARALLEL:1 OLLAMA_ORIGINS:[http://localhost https://localhost http://localhost:* https://localhost:* http://127.0.0.1 https://127.0.0.1 http://127.0.0.1:* https://127.0.0.1:* http://0.0.0.0 https://0.0.0

[GIN] 2026/08/13 - 02:53:38 | 200 |      88.891µs |       127.0.0.1 | GET      "/"
Ollama arriba.


time=2026-08-13T02:53:38.162Z level=INFO source=types.go:32 msg="inference compute" id=0 filter_id=0 library=CUDA compute=7.5 name=CUDA0 description="Tesla T4" libdirs=ollama,cuda_v13 driver=13.0 pci_id=0000:00:04.0 type=discrete total="14.6 GiB" available="14.5 GiB"
time=2026-08-13T02:53:38.162Z level=INFO source=types.go:32 msg="inference compute" id=1 filter_id=1 library=CUDA compute=7.5 name=CUDA1 description="Tesla T4" libdirs=ollama,cuda_v13 driver=13.0 pci_id=0000:00:05.0 type=discrete total="14.6 GiB" available="14.5 GiB"
time=2026-08-13T02:53:38.162Z level=INFO source=routes.go:2040 msg="vram-based default context" total_vram="29.1 GiB" default_num_ctx=32768


In [18]:
# 4. Bajar los modelos
!ollama pull qwen2.5-coder:7b
!ollama pull phi4

]11;?\[GIN] 2026/08/13 - 02:53:43 | 200 |      45.037µs |       127.0.0.1 | HEAD     "/"
pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ [GIN] 2026/08/13 - 02:53:43 | 200 |   764.92773ms |       127.0.0.1 | POST     "/api/pull"
pulling manifest 
pulling 60e05f210007: 100% ▕██████████████████▏ 4.7 GB                         
pulling 66b9ea09bd5b: 100% ▕██████████████████▏   68 B                         
pulling 1e65450c3067: 100% ▕██████████████████▏ 1.6 KB                         
pulling 832dd9e00a68: 100% ▕██████████████████▏  11 KB                         
pulling d9bb33f27869: 100% ▕██████████████████▏  487 B                         
verifying sha256 digest 
writing manifest 
success 
]11;?\[GIN] 2026/08/13 - 02:53:49 | 200 |        43.6µs |       127.0.0.1 | HEAD     "/"
pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ [GIN] 2026

In [19]:
# 5. Instalar OpenEvolve
!pip install -q openevolve

In [20]:
# 6. ZONA DE CONFIGURACION DEL EXPERIMENTO
# ============ CAMBIAR AQUI (1) ============
NOMBRE_EXPERIMENTO = "bin_packing"   # nombre de carpeta bajo examples/
NOMBRE_FUNCION = "empaquetar"        # nombre de la funcion que se evoluciona
# ============================================

import os
CARPETA = f"examples/{NOMBRE_EXPERIMENTO}"
os.makedirs(CARPETA, exist_ok=True)
print("Carpeta del experimento:", CARPETA)

Carpeta del experimento: examples/bin_packing


In [21]:
# 7. Programa semilla
# ============ CAMBIAR AQUI (2) ============
# Logica del algoritmo del que parte la busqueda: First-Fit ingenuo, SIN
# ordenar los items primero (a proposito debil -- ordenar descendente ya
# es una mejora facil de encontrar, y de ahi para adelante hay margen real).
initial_program = f'''
# EVOLVE-BLOCK-START
def {NOMBRE_FUNCION}(items, capacidad):
    """
    Empaqueta una lista de items (cada uno <= capacidad) en la menor
    cantidad posible de contenedores de tamano `capacidad`.

    Devuelve una lista de bins; cada bin es una lista con los tamanos de
    los items que contiene. La suma de cada bin no debe superar `capacidad`,
    y cada item de la entrada debe aparecer exactamente una vez en la salida.

    Implementacion inicial: First-Fit ingenuo, en el orden dado (sin
    ordenar). El objetivo es reescribir esta funcion para usar MENOS
    contenedores en promedio sobre los casos de prueba.
    """
    bins = []
    for item in items:
        colocado = False
        for b in bins:
            if sum(b) + item <= capacidad:
                b.append(item)
                colocado = True
                break
        if not colocado:
            bins.append([item])
    return bins
# EVOLVE-BLOCK-END
'''
# ============================================

with open(f"{CARPETA}/initial_program.py", "w") as f:
    f.write(initial_program)

print(initial_program)


# EVOLVE-BLOCK-START
def empaquetar(items, capacidad):
    """
    Empaqueta una lista de items (cada uno <= capacidad) en la menor
    cantidad posible de contenedores de tamano `capacidad`.

    Devuelve una lista de bins; cada bin es una lista con los tamanos de
    los items que contiene. La suma de cada bin no debe superar `capacidad`,
    y cada item de la entrada debe aparecer exactamente una vez en la salida.

    Implementacion inicial: First-Fit ingenuo, en el orden dado (sin
    ordenar). El objetivo es reescribir esta funcion para usar MENOS
    contenedores en promedio sobre los casos de prueba.
    """
    bins = []
    for item in items:
        colocado = False
        for b in bins:
            if sum(b) + item <= capacidad:
                b.append(item)
                colocado = True
                break
        if not colocado:
            bins.append([item])
    return bins
# EVOLVE-BLOCK-END



In [22]:
# 8. Evaluador
#
# A diferencia del notebook de x^2+y^2, aca NO instrumentamos operaciones
# internas (no hay nada que "contar" en el codigo). El score se calcula
# 100% desde afuera: se corre la funcion candidata, se valida que el
# empaquetamiento sea correcto, y se compara la cantidad de contenedores
# usados contra la cota inferior teorica. Esto cierra la puerta a "hacer
# trampa": la unica forma de mejorar el score es empaquetar mejor de verdad.

evaluator = f'''
"""
Evaluador para el descubrimiento de una heuristica de bin-packing.
Mide solo el resultado (contenedores usados vs. cota inferior teorica).
"""

import importlib.util
import math
import traceback


# ============ CAMBIAR AQUI (3): CASOS DE PRUEBA ============
# Cada caso: (items, capacidad). Fijos y deterministas (sin aleatoriedad).
CASOS_DE_PRUEBA = [
    ([2, 5, 4, 7, 1, 3, 8, 6, 5, 2, 9, 4], 10),
    ([5, 5, 5, 5, 5, 5, 5, 5], 10),
    ([1, 2, 3, 4, 5, 6, 7, 8, 9], 15),
    ([9, 8, 2, 2, 5, 4, 4, 3, 6, 6, 1, 7], 12),
    ([3, 3, 3, 3, 3, 3, 3], 9),
    ([10, 1, 1, 1, 1, 1, 1, 1, 1, 1], 10),
]
# ============================================


def _cargar_funcion(ruta_programa):
    spec = importlib.util.spec_from_file_location("programa_candidato", ruta_programa)
    modulo = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(modulo)
    return getattr(modulo, "{NOMBRE_FUNCION}")


# ============ CAMBIAR AQUI (4): VALIDACION + COTA INFERIOR ============
# Reemplaza al "valor esperado" del caso x^2+y^2: aca no hay una igualdad
# exacta que verificar, sino una restriccion de validez (que el
# empaquetamiento sea legal) y una cota teorica contra la que medir calidad.
def _validar_y_contar(bins, items, capacidad):
    items_usados = []
    for b in bins:
        if sum(b) > capacidad + 1e-9:
            raise ValueError("Un contenedor supera la capacidad")
        items_usados.extend(b)
    if sorted(items_usados) != sorted(items):
        raise ValueError("Los items empaquetados no coinciden con los items de entrada")
    return len(bins)


def _cota_inferior(items, capacidad):
    return math.ceil(sum(items) / capacidad)
# ============================================


def evaluate(program_path):
    try:
        funcion = _cargar_funcion(program_path)
    except Exception:
        return {{"combined_score": 0.0, "error": traceback.format_exc()[:500]}}

    ratios = []
    for items, capacidad in CASOS_DE_PRUEBA:
        try:
            bins = funcion(list(items), capacidad)
            num_bins = _validar_y_contar(bins, items, capacidad)
        except Exception:
            return {{"combined_score": 0.0, "error": traceback.format_exc()[:500]}}

        ratios.append(_cota_inferior(items, capacidad) / num_bins)  # 1.0 = optimo teorico

    # ============ CAMBIAR AQUI (5): FORMULA DE COMBINED_SCORE ============
    combined_score = sum(ratios) / len(ratios)
    # ============================================

    return {{
        "combined_score": combined_score,
        "ratio_promedio": combined_score,
    }}
'''

with open(f"{CARPETA}/evaluator.py", "w") as f:
    f.write(evaluator)

print("evaluator.py escrito (", len(evaluator.splitlines()), "lineas )")

evaluator.py escrito ( 75 lineas )


In [23]:
# 9. Probar el evaluador contra el programa semilla
import sys, importlib

sys.path.insert(0, CARPETA)
if "evaluator" in sys.modules:
    importlib.reload(sys.modules["evaluator"])
import evaluator as ev

print(ev.evaluate(f"{CARPETA}/initial_program.py"))
# El baseline First-Fit sin ordenar deberia dar un ratio_promedio < 1.0,
# con margen claro de mejora.

{'combined_score': 0.9067460317460317, 'ratio_promedio': 0.9067460317460317}


In [24]:
# 10. Config de OpenEvolve
# ============ CAMBIAR AQUI (6): DESCRIPCION DEL OBJETIVO PARA EL LLM ============
system_message = f"""
Eres un experto en algoritmos combinatorios y heuristicas de optimizacion,
en particular bin-packing. Tu tarea es reescribir la funcion
{NOMBRE_FUNCION}(items, capacidad) para que use MENOS contenedores en
promedio que el metodo actual (First-Fit sin ordenar), manteniendo el
empaquetamiento siempre valido: cada item debe usarse exactamente una vez
y ningun contenedor puede superar la capacidad. Podes ordenar los items,
combinar heuristicas conocidas (First-Fit, Best-Fit, First-Fit Decreasing,
etc.) o inventar una variante propia.
"""
# ============================================

config_yaml = f'''
max_iterations: 30
diff_based_evolution: false

llm:
  api_base: "http://127.0.0.1:11434/v1"
  api_key: "ollama"
  primary_model: "qwen2.5-coder:7b"
  secondary_model: "phi4"
  primary_model_weight: 0.7
  secondary_model_weight: 0.3
  temperature: 0.5
  timeout: 180
  retries: 3

prompt:
  system_message: |
{chr(10).join("    " + linea for linea in system_message.strip().split(chr(10)))}

database:
  population_size: 60
  num_islands: 3
  migration_interval: 15

evaluator:
  timeout: 30
'''

with open(f"{CARPETA}/config.yaml", "w") as f:
    f.write(config_yaml)

print(config_yaml)


max_iterations: 30
diff_based_evolution: false

llm:
  api_base: "http://127.0.0.1:11434/v1"
  api_key: "ollama"
  primary_model: "qwen2.5-coder:7b"
  secondary_model: "phi4"
  primary_model_weight: 0.7
  secondary_model_weight: 0.3
  temperature: 0.5
  timeout: 180
  retries: 3

prompt:
  system_message: |
    Eres un experto en algoritmos combinatorios y heuristicas de optimizacion,
    en particular bin-packing. Tu tarea es reescribir la funcion
    empaquetar(items, capacidad) para que use MENOS contenedores en
    promedio que el metodo actual (First-Fit sin ordenar), manteniendo el
    empaquetamiento siempre valido: cada item debe usarse exactamente una vez
    y ningun contenedor puede superar la capacidad. Podes ordenar los items,
    combinar heuristicas conocidas (First-Fit, Best-Fit, First-Fit Decreasing,
    etc.) o inventar una variante propia.

database:
  population_size: 60
  num_islands: 3
  migration_interval: 15

evaluator:
  timeout: 30



In [25]:
# 11. Correr OpenEvolve
!openevolve-run {CARPETA}/initial_program.py \
    {CARPETA}/evaluator.py \
    --config {CARPETA}/config.yaml \
    --iterations 10

2026-08-13 02:53:54,706 - INFO - Logging to examples/bin_packing/openevolve_output/logs/openevolve_20260813_025354.log
2026-08-13 02:53:54,718 - INFO - Set random seed to 42 for reproducibility
2026-08-13 02:53:54,798 - INFO - Initialized OpenAI LLM with model: qwen2.5-coder:7b
2026-08-13 02:53:54,839 - INFO - Initialized OpenAI LLM with model: phi4
2026-08-13 02:53:54,839 - INFO - Initialized LLM ensemble with models: qwen2.5-coder:7b (weight: 0.70), phi4 (weight: 0.30)
2026-08-13 02:53:54,918 - INFO - Initialized LLM ensemble with models: qwen2.5-coder:7b (weight: 0.70), phi4 (weight: 0.30)
2026-08-13 02:53:54,919 - INFO - Initialized prompt sampler
2026-08-13 02:53:54,920 - INFO - Set custom templates: system=evaluator_system_message, user=None
2026-08-13 02:53:54,920 - INFO - Initialized program database with 0 programs
2026-08-13 02:53:54,921 - INFO - Successfully loaded evaluation function from examples/bin_packing/evaluator.py
2026-08-13 02:53:54,921 - WARNING - Configuration ha

time=2026-08-13T02:53:57.890Z level=INFO source=sched.go:1013 msg="selecting single GPU for llama-server model" main_gpu=0 id=0 filter_id=0 library=CUDA name=CUDA0 description="Tesla T4" integrated=false predicted="11.6 GiB" available="14.5 GiB"
time=2026-08-13T02:53:57.890Z level=INFO source=sched.go:1132 msg="selecting GPU backend for llama-server model" library=CUDA gpu_count=1 available_gpu_count=2
time=2026-08-13T02:53:57.890Z level=INFO source=sched.go:1215 msg="disabling mmap for llama-server load due to host memory pressure" model=/root/.ollama/models/blobs/sha256-fd7b6731c33c57f61767612f56517460ec2d1e2e5a3f0163e0eb3d8d8cb5df20 model_size="8.4 GiB" loaded_mmap_size="0 B" headroom="7.5 GiB" system_free="13.0 GiB" system_total="30.0 GiB" predicted_vram="11.6 GiB" available_vram="14.5 GiB"
time=2026-08-13T02:53:57.890Z level=INFO source=server.go:109 msg="using llama-server for model" model=/root/.ollama/models/blobs/sha256-fd7b6731c33c57f61767612f56517460ec2d1e2e5a3f0163e0eb3d8d8

[GIN] 2026/08/13 - 02:54:42 | 200 | 47.577661111s |       127.0.0.1 | POST     "/v1/chat/completions"
2026-08-13 02:54:42,923 - INFO - HTTP Request: POST http://127.0.0.1:11434/v1/chat/completions "HTTP/1.1 200 OK"
[[8, 2], [4, 4, 1, 1]]
2026-08-13 02:54:42,934 - INFO - Evaluated program 2ee74fdd-995e-4ed3-86bc-48d424f987aa in 0.00s: combined_score=1.0000, ratio_promedio=1.0000
2026-08-13 02:54:42,936 - INFO - Sampled model: phi4
2026-08-13 02:54:42,945 - INFO - New MAP-Elites cell occupied in island 0: {'complexity': 9, 'diversity': 5}
2026-08-13 02:54:42,945 - INFO - New best program 2ee74fdd-995e-4ed3-86bc-48d424f987aa replaces a2b06440-d579-4967-9390-7c908972ef15 (combined_score: 0.9067 → 1.0000, +0.0933)
2026-08-13 02:54:42,945 - INFO - Iteration 1: Program 2ee74fdd-995e-4ed3-86bc-48d424f987aa (parent: a2b06440-d579-4967-9390-7c908972ef15) completed in 47.84s
2026-08-13 02:54:42,945 - INFO - Metrics: combined_score=1.0000, ratio_promedio=1.0000
2026-08-13 02:54:42,945 - INFO - 🌟 N

slot print_timing: id  0 | task 0 | prompt eval time =    1501.45 ms /   851 tokens (    1.76 ms per token,   566.78 tokens per second)
slot print_timing: id  0 | task 0 |        eval time =   40723.40 ms /   508 tokens (   80.16 ms per token,    12.47 tokens per second)
slot print_timing: id  0 | task 0 |       total time =   42224.86 ms /  1359 tokens
slot print_timing: id  0 | task 0 |    graphs reused =        505
slot      release: id  0 | task 0 | stop processing: n_tokens = 1358, truncated = 0
srv  update_slots: all slots are idle
srv  server_strea: conv_id= (empty=1)
slot get_availabl: id  0 | task -1 |  - checking sim = 1.000 (851/851) > 0.100
slot get_availabl: id  0 | task -1 | selected slot by LCP similarity, f_sim_best = 1.000 (> 0.100 thold), f_keep = 0.627
slot launch_slot_: id  0 | task -1 | sampler chain: logits -> penalties -> ?dry -> ?top-n-sigma -> top-k -> ?typical -> ?top-p -> ?min-p -> ?xtc -> temp-ext -> dist 
slot launch_slot_: id  0 | task -1 | sampler params:

[GIN] 2026/08/13 - 02:55:29 | 200 | 46.858927002s |       127.0.0.1 | POST     "/v1/chat/completions"
2026-08-13 02:55:29,797 - INFO - HTTP Request: POST http://127.0.0.1:11434/v1/chat/completions "HTTP/1.1 200 OK"
2026-08-13 02:55:29,800 - INFO - Evaluated program e4839d3b-498a-4110-9bf7-4cb8d61474fe in 0.00s: combined_score=1.0000, ratio_promedio=1.0000
2026-08-13 02:55:29,802 - INFO - Sampled model: qwen2.5-coder:7b
2026-08-13 02:55:29,802 - INFO - New MAP-Elites cell occupied in island 1: {'complexity': 1, 'diversity': 0}
2026-08-13 02:55:29,802 - INFO - Iteration 2: Program e4839d3b-498a-4110-9bf7-4cb8d61474fe (parent: a2b06440-d579-4967-9390-7c908972ef15) completed in 46.87s
2026-08-13 02:55:29,802 - INFO - Metrics: combined_score=1.0000, ratio_promedio=1.0000


...

slot print_timing: id  0 | task 2315 | prompt eval time =    1910.11 ms /  1044 tokens (    1.83 ms per token,   546.57 tokens per second)
slot print_timing: id  0 | task 2315 |        eval time =   78584.01 ms /   877 tokens (   89.61 ms per token,    11.16 tokens per second)
slot print_timing: id  0 | task 2315 |       total time =   80494.11 ms /  1921 tokens
slot print_timing: id  0 | task 2315 |    graphs reused =       3163
slot      release: id  0 | task 2315 | stop processing: n_tokens = 2131, truncated = 0
srv  update_slots: all slots are idle
srv  server_strea: conv_id= (empty=1)
slot get_availabl: id  0 | task -1 |  - checking sim = 0.109 (156/1426) > 0.100
slot get_availabl: id  0 | task -1 | selected slot by LCP similarity, f_sim_best = 0.109 (> 0.100 thold), f_keep = 0.069
srv  get_availabl: updating prompt cache
srv   prompt_save:  - saving prompt with length 2264, total state size = 123.839 MiB (draft: 0.000 MiB)
srv          load:  - looking for better prompt, base f_k

[GIN] 2026/08/13 - 03:00:03 | 200 | 18.147077001s |       127.0.0.1 | POST     "/v1/chat/completions"
2026-08-13 03:00:03,193 - INFO - HTTP Request: POST http://127.0.0.1:11434/v1/chat/completions "HTTP/1.1 200 OK"
[[4, 1], [3, 2]]
2026-08-13 03:00:03,196 - INFO - Evaluated program 879369c0-9b7d-4eda-bf4b-fb50ac7ad9cd in 0.00s: combined_score=1.0000, ratio_promedio=1.0000
2026-08-13 03:00:03,198 - INFO - Sampled model: phi4
2026-08-13 03:00:03,204 - INFO - New MAP-Elites cell occupied in island 2: {'complexity': 7, 'diversity': 9}
2026-08-13 03:00:03,204 - INFO - Iteration 9: Program 879369c0-9b7d-4eda-bf4b-fb50ac7ad9cd (parent: 9bb08a4b-5f8d-4224-b7bc-b751f3c093df) completed in 18.15s
2026-08-13 03:00:03,204 - INFO - Metrics: combined_score=1.0000, ratio_promedio=1.0000


slot print_timing: id  0 | task 1860 | prompt eval time =    1070.46 ms /  1270 tokens (    0.84 ms per token,  1186.41 tokens per second)
slot print_timing: id  0 | task 1860 |        eval time =   16661.07 ms /   613 tokens (   27.18 ms per token,    36.79 tokens per second)
slot print_timing: id  0 | task 1860 |       total time =   17731.53 ms /  1883 tokens
slot print_timing: id  0 | task 1860 |    graphs reused =       2453
slot      release: id  0 | task 1860 | stop processing: n_tokens = 2038, truncated = 0
srv  update_slots: all slots are idle
srv  server_strea: conv_id= (empty=1)
slot get_availabl: id  0 | task -1 |  - checking sim = 0.077 (158/2051) > 0.100
slot get_availabl: id  0 | task -1 | selected slot by LRU, t_last = 1572941783
srv  get_availabl: updating prompt cache
srv   prompt_save:  - saving prompt with length 2131, total state size = 416.236 MiB (draft: 0.000 MiB)
srv          load:  - looking for better prompt, base f_keep = 0.074, f_sim = 0.077
srv          lo

[GIN] 2026/08/13 - 03:01:19 | 200 |         1m16s |       127.0.0.1 | POST     "/v1/chat/completions"
2026-08-13 03:01:19,442 - INFO - HTTP Request: POST http://127.0.0.1:11434/v1/chat/completions "HTTP/1.1 200 OK"
[[8, 2], [4, 4, 1, 1]]
2026-08-13 03:01:19,445 - INFO - Evaluated program b8feacfa-e2d6-417c-b59c-60cd4994661d in 0.00s: combined_score=1.0000, ratio_promedio=1.0000
2026-08-13 03:01:19,447 - INFO - Iteration 10: Program b8feacfa-e2d6-417c-b59c-60cd4994661d (parent: a2b06440-d579-4967-9390-7c908972ef15) completed in 76.25s
2026-08-13 03:01:19,448 - INFO - Metrics: combined_score=1.0000, ratio_promedio=1.0000
2026-08-13 03:01:19,448 - INFO - ✅ Evolution completed - Maximum iterations reached
2026-08-13 03:01:19,448 - INFO - Received signal 15, initiating graceful shutdown...
2026-08-13 03:01:19,449 - INFO - Graceful shutdown requested...
2026-08-13 03:01:19,455 - INFO - Stopped process pool
2026-08-13 03:01:19,456 - INFO - Using tracked best program: 2ee74fdd-995e-4ed3-86bc-4

slot print_timing: id  0 | task 3195 | prompt eval time =    3390.73 ms /  1893 tokens (    1.79 ms per token,   558.29 tokens per second)
slot print_timing: id  0 | task 3195 |        eval time =   72257.21 ms /   798 tokens (   90.55 ms per token,    11.04 tokens per second)
slot print_timing: id  0 | task 3195 |       total time =   75647.94 ms /  2691 tokens
slot print_timing: id  0 | task 3195 |    graphs reused =       3956
slot      release: id  0 | task 3195 | stop processing: n_tokens = 2848, truncated = 0
srv  update_slots: all slots are idle


In [26]:
# 12. Ver la mejor heuristica encontrada
mejor_path = f"{CARPETA}/openevolve_output/best/best_program.py"
with open(mejor_path) as f:
    print(f.read())

import evaluator as ev
importlib.reload(ev)
print(ev.evaluate(mejor_path))

def empaquetar(items, capacidad):
    """
    Empaqueta una lista de items (cada uno <= capacidad) en la menor
    cantidad posible de contenedores de tamano `capacidad`.

    Devuelve una lista de bins; cada bin es una lista con los tamanos de
    los items que contiene. La suma de cada bin no debe superar `capacidad`,
    y cada item de la entrada debe aparecer exactamente una vez en la salida.

    Implementacion: First-Fit Decreasing.
    """
    # Sort the items in decreasing order
    sorted_items = sorted(items, reverse=True)
    
    bins = []
    
    for item in sorted_items:
        colocado = False
        
        # Try to place the item in an existing bin
        for b in bins:
            if sum(b) + item <= capacidad:
                b.append(item)
                colocado = True
                break
        
        # If it doesn't fit in any existing bin, create a new one
        if not colocado:
            bins.append([item])
    
    return bins

# Example usage
i

In [27]:
# 13. Graficar la evolucion de la busqueda
# (todas las evaluaciones individuales + mejor score encontrado hasta el
# momento, igual que un grafico tipico de "evolution of the best candidate")
import glob, json
import matplotlib.pyplot as plt


def _recolectar_programas(carpeta):
    """Recorre TODOS los checkpoints y junta los programas evaluados
    (deduplicados por id), con su iteracion, score y codigo fuente."""
    programas = {}
    patron = f"{carpeta}/openevolve_output/checkpoints/checkpoint_*/programs/*.json"
    for ruta in glob.glob(patron):
        try:
            with open(ruta) as f:
                data = json.load(f)
        except Exception:
            continue
        pid = data.get("id", ruta)
        metricas = data.get("metrics", {}) or {}
        score = metricas.get("combined_score")
        if score is None:
            continue
        programas[pid] = {
            "iteracion": data.get("iteration_found", 0),
            "combined_score": score,
            "codigo": data.get("code", ""),
        }
    return programas


programas = _recolectar_programas(CARPETA)
puntos = sorted(programas.values(), key=lambda p: p["iteracion"])

if not puntos:
    print("No se encontraron programas evaluados todavia. Corre primero la celda 11.")
else:
    iteraciones = [p["iteracion"] for p in puntos]
    scores = [p["combined_score"] for p in puntos]

    mejor_hasta_ahora = []
    mejor = float("-inf")
    for s in scores:
        mejor = max(mejor, s)
        mejor_hasta_ahora.append(mejor)

    baseline_score = ev.evaluate(f"{CARPETA}/initial_program.py")["combined_score"]

    plt.figure(figsize=(9, 5))
    plt.scatter(iteraciones, scores, color="gray", alpha=0.5, label="Evaluaciones individuales")
    plt.plot(iteraciones, mejor_hasta_ahora, color="red", linewidth=2,
              label="Mejor score encontrado hasta el momento")
    plt.axhline(baseline_score, color="black", linestyle="--", linewidth=1,
                 label="Baseline (First-Fit sin ordenar)")
    plt.title("Evolucion del mejor candidato durante la busqueda")
    plt.xlabel("Iteracion (segun checkpoint)")
    plt.ylabel("Combined score")
    plt.legend()
    plt.tight_layout()
    plt.show()

No se encontraron programas evaluados todavia. Corre primero la celda 11.


In [28]:
# 14. Imprimir por consola TODOS los programas generados
# (requiere haber corrido la celda 13 antes, para tener `programas`)
for p in sorted(programas.values(), key=lambda p: p["iteracion"]):
    print("=" * 80)
    print(f"Iteracion: {p['iteracion']}  |  combined_score: {p['combined_score']:.4f}")
    print("-" * 80)
    print(p["codigo"])
    print()